Bitcraze Crazyflie 2.1+ Test Notebook
=====================================

Platform specification
----------------------

**OS, OS release, Python version, Compiler version**

In [1]:
!uname -a
!lsb_release -d
!python --version
!gcc --version | grep gcc
!influxd version
!telegraf version
# !curl -s http://localhost:8086/health | grep 'version'

Linux edge-device-2 6.1.21-v8+ #1642 SMP PREEMPT Mon Apr  3 17:24:16 BST 2023 aarch64 GNU/Linux
Description:	Debian GNU/Linux 11 (bullseye)
Python 3.12.8
gcc (Debian 10.2.1-6) 10.2.1 20210110
InfluxDB v2.9.1 (git: d4fa1941fd) build_date: 2026-05-11T20:48:40Z
Telegraf 1.40.1 (git: HEAD@26b8f447)


**Dependencies**

In [2]:
!echo -e "crazyflie-lib-python: " $(python -m pip list | grep "cflib")
!echo -e "dotenv: " $(python -m pip list | grep "dotenv")
!echo -e "influxDB: " $(python -m pip list | grep "influx")

crazyflie-lib-python:  cflib 0.1.33
dotenv:  dotenv 0.9.9 python-dotenv 1.2.3
influxDB:  influxdb_client 1.50.0


# 1. Initialization

In [3]:
import cflib.drivers.crazyradio as crazyradio   # Communication medium device
import cflib.crtp                               # Communication protocol
from cflib.crazyflie import Crazyflie           # Drone class
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie # Drone class for synchronized tasks
from cflib.crazyflie.log import LogConfig
from cflib.crazyflie.syncLogger import SyncLogger
from cflib.positioning.motion_commander import MotionCommander
from cflib.positioning.position_hl_commander import PositionHlCommander
from cflib.utils import uri_helper
from cflib.utils.reset_estimator import reset_estimator

import bisect
from collections import defaultdict
from dotenv import load_dotenv, dotenv_values
import csv
import logging
import math
import os
import sys
import struct
import threading
from threading import Event, Lock
import time # time.monotonic
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output

# InfluxDB
import influxdb_client
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

**Environment Variables**

In [4]:
load_dotenv()  # reads variables from a .env file and sets them in os.environ
config = dotenv_values(".env")
#print(config['INFLUXDB_BUCKET'])

# URI to the Crazyflie to connect to
URIs = []
for d in range(1,4):
    URIs.append( uri_helper.uri_from_env(env=f'DRONE{str(d)}_URI', default='radio://0/80/2M/E7E7E7E7E7') )

**Services**

In [5]:
# Start InfluxDB
# Telegraf: Create a configuration file
# Telegraf: Start the service
print( "InfluxDB - Host:", os.getenv("INFLUXDB_HOST") )
print( "InfluxDB - Port:", os.getenv("INFLUXDB_PORT") )
print( "InfluxDB - URL:", os.getenv("INFLUXDB_URL") )
print( "InfluxDB - Organization:", os.getenv("INFLUXDB_ORG") )
print( "InfluxDB - Token:", os.getenv("INFLUXDB_TOKEN") )
print( "InfluxDB - Bucket:", os.getenv("INFLUXDB_BUCKET") )

try:
    db_client = influxdb_client.InfluxDBClient(
        url=os.getenv("INFLUXDB_URL"),
        username=os.getenv("INFLUXDB_USERNAME"),
        password=os.getenv("INFLUXDB_PASSWORD"),
        org=os.getenv("INFLUXDB_ORG"),
        #token=os.getenv("INFLUXDB_TOKEN"),
        ssl=False,
        verify_ssl=False,
    )

    # Check connection
    if not db_client.ping():
        raise Exception("Service not available")
    else:
        print("Connection established to InfluxDB")

    # Initialize query API
    query_api = db_client.query_api()
    # Initialize client write API
    write_api = db_client.write_api(write_options=SYNCHRONOUS)

    # Check database
    
    #p = influxdb_client.Point("my_measurement").tag("location", "Prague").field("temperature", 25.3)
    #write_api.write(bucket=bucket, org=org, record=p)
    
    
except Exception as e:
    print(e)

InfluxDB - Host: http://localhost
InfluxDB - Port: 8086
InfluxDB - URL: http://localhost:8086
InfluxDB - Organization: bitcraze
InfluxDB - Token: JSCkOys-pxxx-qFFWKymREPtPDValrUuF9qcieOC9vBxfI6lW3rmiBVHPW1-iRS3WjSXpCLfDLjME-W60-98kg==
InfluxDB - Bucket: crazyflie-bucket
Connection established to InfluxDB


**Configuration**

In [6]:
#format = "%(asctime)s.%(msecs)03d - %(message)s"
#logging.basicConfig(format=format, level=logging.INFO, datefmt="%H:%M:%S")
#logging.basicConfig(level=logging.ERROR)
deck_attached_event = Event()
crazyflie_ready = Event()

**InfluxDB Configuration**

In [7]:
# Check or create a database

# Check or create a table

# 2. Logging

In [8]:
print("Python Version", sys.version)

Python Version 3.12.8 (main, Jun 17 2026, 18:05:58) [GCC 10.2.1 20210110]


## 2.1 Logging configurations

In [9]:
move = [False, False, False, False] # Forward, Backward, Left, Right

def log_configurations():
    logconf = []
    # Radio signal
    def log_radio_conf():
        log_conf = LogConfig(name="Radio", period_in_ms=500)
        log_conf.add_variable('radio.rssi', 'uint8_t')        # Radio Signal Strength Indicator [dBm]
        log_conf.add_variable('radio.isConnected', 'uint8_t') # Indicator if a packet was received from the radio within the last RADIO_ACTIVITY_TIMEOUT_MS
        log_conf.add_variable('radio.numRxBc', 'uint16_t')    # Number of broadcast packets received
        log_conf.add_variable('radio.numRxUc', 'uint16_t')    # Number of unicast packets received
        return log_conf
    # Stabilizer
    def log_stab_conf():
        log_conf = LogConfig(name='Stabilizer', period_in_ms=10) # 100 Hz
        log_conf.add_variable('stabilizer.roll', 'float')
        log_conf.add_variable('stabilizer.pitch', 'float')
        log_conf.add_variable('stabilizer.yaw', 'float')
        return log_conf
    # Attitude
    def log_att_conf():
        log_conf = LogConfig(name='Attitude', period_in_ms=200)
        log_conf.add_variable('stateEstimate.roll', 'float')
        log_conf.add_variable('stateEstimate.pitch', 'float')
        log_conf.add_variable('stateEstimate.yaw', 'float')
        return log_conf
    # Position
    def log_pos_conf():
        log_conf = LogConfig(name='Position', period_in_ms=200)
        log_conf.add_variable('kalman.stateX', 'float')
        log_conf.add_variable('kalman.stateY', 'float')
        log_conf.add_variable('kalman.stateZ', 'float')
        return log_conf
    
    def log_pos_state_conf():
        log_conf = LogConfig(name='State', period_in_ms=50)
        log_conf.add_variable('stateEstimate.x', 'float')
        log_conf.add_variable('stateEstimate.y', 'float')
        log_conf.add_variable('stateEstimate.z', 'float')
        log_conf.add_variable('stateEstimate.vx', 'float')
        log_conf.add_variable('stateEstimate.vy', 'float')
        log_conf.add_variable('stateEstimate.yaw', 'float')
        return log_conf
    # Multi-ranger deck
    def log_range_conf():
        log_conf = LogConfig(name='Range', period_in_ms=200)
        log_type = 'uint16_t' # Distance from the sensor to an obstacle in mm
        log_conf.add_variable('range.front', 'uint16_t')
        log_conf.add_variable('range.back', 'uint16_t')
        log_conf.add_variable('range.left', 'uint16_t')
        log_conf.add_variable('range.right', 'uint16_t')
        log_conf.add_variable('range.up', 'uint16_t')
        log_conf.add_variable('range.zrange', 'uint16_t')
        return log_conf
    # Battery
    def log_battery_conf():
        log_conf = LogConfig(name='Battery', period_in_ms=1000)
        log_conf.add_variable("pm.vbat", 'float') # FP16
        log_conf.add_variable("pm.vbatMV", 'uint16_t')
        log_conf.add_variable("pm.chargeCurrent", 'float')
        log_conf.add_variable("pm.batteryLevel", 'uint8_t')
        return log_conf
    # Battery - Health Test
    def log_battery_health_conf():
        log_conf = LogConfig(name='BatteryHealth', period_in_ms=1000)
        log_conf.add_variable("health.batterySag", 'float')
        log_conf.add_variable("health.batteryPass", 'uint8_t')
        return log_conf
    # Supervisor
    def log_supervisor_conf():
        log_conf = LogConfig(name='Supervisor', period_in_ms=500)
        log_conf.add_variable('supervisor.info', 'uint16_t')
        return log_conf
    
    def log_sys_conf():
        log_conf = LogConfig(name='Sys', period_in_ms=500)
        log_conf.add_variable('sys.isFlying', 'uint8_t')
        return log_conf
        
    def log_imu_conf():
        log_conf = LogConfig(name='Position', period_in_ms=30)
        imu_type="FP16"
        log_conf.add_variable("acc.x", imu_type)
        log_conf.add_variable("acc.y", imu_type)
        log_conf.add_variable("acc.z", imu_type)
        log_conf.add_variable("gyro.x", imu_type)
        log_conf.add_variable("gyro.y", imu_type)
        log_conf.add_variable("gyro.z", imu_type)

    # For Health Tests -------------------------
    #logconf.append( log_battery_health_conf() )
    # ------------------------------------------

    # For Mission ------------------------------
    #logconf.append( log_radio_conf() )
    logconf.append( log_pos_state_conf() )
    logconf.append( log_range_conf() )
    logconf.append( log_battery_conf() )
    #logconf.append( log_sys_conf() )
    #logconf.append( log_supervisor_conf() )
    # ------------------------------------------

    return logconf
# Each logconf has these methods and properties: 'add_memory', 'add_variable', 'added', 'added_cb', 'cf', 
# \ 'create', 'data_received_cb', 'default_fetch_as', 'delete', 'err_no', 'error_cb', 'id', 'name', 'pending', 
# \ 'period', 'period_in_ms', 'start', 'started', 'started_cb', 'stop', 'unpack_log_data', 'useV2', 'valid', 'variables'

## 2.2 Logging callbacks

In [10]:
os.makedirs('logs', exist_ok=True)
path = os.path.join('logs', f"flight_{datetime.now():%Y%m%d_%H%M%S}.csv")


In [11]:
FIELDS = [
    "t_host_s", "t_fw_ms", "phase",
    "x_m", "y_m", "z_m", "vx_ms", "vy_ms", "yaw_deg",
    "front_mm", "back_mm", "left_mm", "right_mm", "up_mm", "down_mm",
    "vbat_v", "battery_pct_est", "battery_level"
]

In [12]:
class Recorder:
    """Receives log packets, merges them and writes one CSV row per state packet."""
    def __init__(self, path):
        self._f = open(path, "w", newline="")
        self._w = csv.writer(self._f)
        self._w.writerow(FIELDS)
        self._lock = threading.Lock()
        self._rng = {}
        #self._bat = 0
        self._vbat = 0.0
        self._closed = False

        # Flags
        self.flag_header = False
        self.flag_print = False

        self.x = self.y = self.yaw = 0.0
        self.battery_level = 0
        self.position_estimate = [0.0, 0.0]
        self.link_quality = 0
        self.radio_latency = []
        self.rssi_logs = []
        #self.ranges = {}
        self.ranges_collection = defaultdict(list)
        self.phase = "init"
        self.t0 = time.monotonic()  # monotonic: immune to NTP/clock jumps

    def radio_stats_cb(self, timestamp, data, logconf):
        self.link_quality = data

    def radio_latency_cb(self, latency):
        self.radio_latency.append( round(latency, 3) )
        # print(f"Latency: {latency:.3f} ms")

    def log_radio_cb(self, timestamp, data, logconf):
        if self.flag_print:
            if not self.flag_header:
                print("|-" + " RADIO LOG " + "-" * 20)
                print("| ", end='')
                print(f"Timestamp", end=' |\t')
                for key in data.keys():
                    print(f"{key}", end=' |\t')
                print("")
                flag_header = True
            print("| ", end='')
            
            print(f"{timestamp}", end=f'{' '*(12-1)}|\t')
            for key, value in data.items():
                self.rssi_logs.append(value) if key == 'radio.rssi' else None
                print(f"{value}", end=f'{' '*(len(key)-1)}|\t')
            print("")
        else:
            self.rssi_logs.append(data['radio.rssi'])
    
    # Multi-Ranger Logging Callback
    @property
    def ranges(self):
        return self._rng
        
    def log_range_cb(self, timestamp, data, logconf):
        """
        print('[%d][%s]: %s' % (timestamp, logconf.name, data))
        t_range.append(timestamp)
        self.ranges_collection['range.front'].append(data['range.front'])
        self.ranges_collection['range.right'].append(data['range.right'])
        self.ranges_collection['range.back'].append(data['range.back'])
        self.ranges_collection['range.left'].append(data['range.left'])
        self.ranges_collection['range.zrange'].append(data['range.zrange'])
        """
        with self._lock:
            self._rng = dict(data)

    def log_pos_state_cb(self, timestamp, data, logconf):
        #self.position_estimate[0] = data['stateEstimate.x']
        #self.position_estimate[1] = data['stateEstimate.y']
        with self._lock:
            if self._closed:
                return
            self.x = data['stateEstimate.x']
            self.y = data['stateEstimate.y']
            self.yaw = data['stateEstimate.yaw']
            r = self._rng # Range
            vb = self._vbat # Battery
            bt = self.battery_level
            self._w.writerow([
                round(time.monotonic() - self.t0, 4), timestamp, self.phase,
                data["stateEstimate.x"], data["stateEstimate.y"], data["stateEstimate.z"],
                data["stateEstimate.vx"], data["stateEstimate.vy"], data["stateEstimate.yaw"],
                r.get("range.front"), r.get("range.back"), r.get("range.left"),
                r.get("range.right"), r.get("range.up"), r.get("range.zrange"),
                vb, self.vbat_to_pct(vb), bt
            ])
                
    # Battery Logging Callback
    # Approximate 1S LiPo open-circuit voltage -> state of charge.
    # Only an estimate: voltage sags under load, so trust it at rest, not in flight.
    _V = [3.27, 3.61, 3.69, 3.71, 3.73, 3.75, 3.77, 3.79, 3.80, 3.82, 3.84, 3.85, 3.87, 3.91, 3.95, 3.98, 4.02, 4.08, 4.11, 4.15, 4.20]
    _P = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
    
    @classmethod
    def vbat_to_pct(cls, v):
        if v is None:
            return None
        if v <= cls._V[0]:
            return 0.0
        if v >= cls._V[-1]:
            return 100.0
        i = bisect.bisect_right(cls._V, v)
        v0, v1, p0, p1 = cls._V[i - 1], cls._V[i], cls._P[i - 1], cls._P[i]
        return round(p0 + (p1 - p0) * (v - v0) / (v1 - v0), 1)
    
    @property
    def battery(self):
        return self._vbat
    
    def log_battery_cb(self, timestamp, data, logconf):
        #self.battery_percentage.append(data['pm.batteryLevel'])
        #print(f"Battery percentage: {self.battery_percentage[-1]}%", end="\r")
        with self._lock:
            self._vbat = data['pm.vbat']
            self.battery_level = data['pm.batteryLevel']
            #self._bat = max(0, min(100, (vbat - 3.0) / (4.2 - 3.0) * 100))
    
    def battery_temp_cb(self, timestamp, data, logconf):
        print(f'[{timestamp}][{logconf.name}]: ', end='')
        for name, value in data.items():
            print(f'{name}: {value:3.3f} ', end='')
        print()
        
    def battery_health_cb(self, timestamp, data, logconf):
        print(f"{timestamp} - {logconf.name} - {data}")

    def log_error_cb(self, logconf, msg):
        print("Error when logging %s" % logconf.name)

    def reset(self, path):
        self._f = open(path, "w", newline="")
        self._w = csv.writer(self._f)
        self._w.writerow(FIELDS)
        self._lock = threading.Lock()
        self._rng = {}
        #self._bat = 0
        self._vbat = 0.0
        self._closed = False

        # Flags
        self.flag_header = False
        self.flag_print = False

        self.x = self.y = self.yaw = 0.0
        self.battery_level = 0
        self.position_estimate = [0.0, 0.0]
        self.link_quality = 0
        self.radio_latency = []
        self.rssi_logs = []
        #self.ranges = {}
        self.ranges_collection = defaultdict(list)
        self.phase = "init"
        self.t0 = time.monotonic()  # monotonic: immune to NTP/clock jumps
        
    def close(self):
        with self._lock:
            if not self._closed:
                self._closed = True
                self._f.flush()
                self._f.close()

In [13]:
rec = Recorder(path)

# 3. Connection

In [14]:
try: 
    for log_conf in log_configs:
        log_conf.stop()
except:
    pass
try:
    scf.close_link()
except:
    pass

In [15]:
# CrazyRadio (Communication Medium) - Dongles currently connected to the USB Port of the computer
radio = crazyradio.Crazyradio()
print(f"|- Crazyradio 2.0 Version: {crazyradio.Crazyradio().version}")
print("\n|- Crazyradio Devices ---" + "-" * 10)
dongles = []
for d in crazyradio._find_devices():
    print(f"|* {d.manufacturer}, {d.product} (Serial number: {d.serial_number})")
    dongles.append(d)
print("|------------------------" + "-" * 10)

# Communication protocol driver - Initialize the low-level drivers
cflib.crtp.init_drivers(enable_serial_driver=False)

interfaces = cflib.crtp.get_interfaces_status()
print(f"|- Radio Interface: {interfaces['radio']}")

# CrazyRadio - Interface scanning
print(f"|- Wait for Crazyflie scanning...", end=' ')
drones = dict() # Dictionary: { 'drone_uri': driverClass_uri}
for uri in URIs:
    drone_addr = uri.split('/')[-1]
    available = cflib.crtp.scan_interfaces(address=int(drone_addr, 16))
    if available:
        #drones.add(available[0][0])
        """
        drones[available[0][0]] = cflib.crtp.get_link_driver(
            uri=available[0][0], 
            #radio_link_statistics_callback=radio_stats_cb,
            link_error_callback=link_error_cb
        )
        """
        drones[available[0][0]] = 1
drones_uri = list(drones.keys())
print("Crazyflie available: ", drones_uri)

|- Crazyradio 2.0 Version: 5.0

|- Crazyradio Devices -------------
|* Bitcraze AB, Crazyradio 2.0 (Serial number: 59C39A09484EB47A)
|----------------------------------
|- Radio Interface: Crazyradio version 5.0
|- Wait for Crazyflie scanning... Crazyflie available:  ['radio://0/80/2M/E7E7E7E702']


## 3.1 Crazyflie Synchronization

In [16]:
drone_uri = drones_uri[0]
print("|- Link", drones[drone_uri])

cf_stats = Crazyflie(rw_cache='./cache')

if not drone_uri:
    print("No drone_uri")
    sys.exit()
    
print(f"|- Wait for Crazyflie Synchronization...", end=' ')
scf = SyncCrazyflie(drone_uri, cf=cf_stats)
scf.open_link()
print("Link opened")

print(f"|- Connected to Crazyflie via {scf.cf.link_uri}")
time.sleep(3)  # give some time to establish connection; 3 sec seems to work
    
print("|- Crazyflie - Wait for params..", end=' ')
scf.wait_for_params()
print("Ok")

|- Link 1
|- Wait for Crazyflie Synchronization... Link opened
|- Connected to Crazyflie via radio://0/80/2M/E7E7E7E702
|- Crazyflie - Wait for params.. Ok


## Health Tests

In [17]:
print("-"*30 + "\n TESTS \n" + "-"*30)
try:
    # Battery test
    battery_health_log = None
    if not battery_health_log:
        battery_health_log = rec.log_battery_health_conf()
        scf.cf.log.add_config(battery_health_log)
        battery_health_log.data_received_cb.add_callback(rec.battery_health_cb)
        battery_health_log.start()
    
    param_startBatTest = scf.cf.param.get_value(complete_name="health.startBatTest")
    scf.cf.param.set_value(complete_name="health.batTestPWMRatio", value=0) # value goes from 0 to 65535, over 2500 propellers spins. Not use 65535
    print(f"|- Battery test: {param_startBatTest} > ", end='')
    if param_startBatTest == str(0):
        scf.cf.param.set_value(complete_name="health.startBatTest", value=1)
        param_batTestPWMRatio = scf.cf.param.get_value(complete_name="health.batTestPWMRatio")
        print(f"1 - Done, Tested with {param_batTestPWMRatio} PWM ratio")
    else:
        scf.cf.param.set_value(complete_name="health.startBatTest", value=0)
        print(f"0 - Skip")
    
    # Power management
    param_lowVoltage = scf.cf.param.get_value(complete_name="pm.lowVoltage")
    print(f"|- Low Voltage threshold: {param_lowVoltage} > ")
    
    param_criticalLowVoltage = scf.cf.param.get_value(complete_name="pm.criticalLowVoltage")
    print(f"|- Critical Low Voltage threshold: {param_criticalLowVoltage} > ")
except:
    logging.warning("Health test skipped")

------------------------------
 TESTS 
------------------------------


## 3.2 Load Logging Configuration

In [18]:
# ---[ Logs configuration ]---
print("|- 2. Log Configurations ", end=': ')
log_configs = log_configurations()
print([log_conf.name for log_conf in log_configs])

|- 2. Log Configurations : ['State', 'Range', 'Battery']


## 3.3 Load callback functions for logging

In [19]:
# Log configuration to logging framework
for log_conf in log_configs:
    
    scf.cf.log.add_config(log_conf)

    if log_conf.valid:
        print(f"|- Loading {log_conf.name}")

        #scf.cf.link_statistics.link_quality_updated.add_callback(radio_stats_cb)
        #latency_callback = lambda latency: print(f"Latency: {latency:.3f} ms")
        #scf.cf.link_statistics.latency.latency_updated.add_callback(radio_latency_cb)
        
        match(log_conf.name):
            case "Radio":
                log_conf.data_received_cb.add_callback(rec.log_radio_cb)
            case "State":
                log_conf.data_received_cb.add_callback(rec.log_pos_state_cb)
            case "Position":
                pass#log_conf.data_received_cb.add_callback(rec.log_pos_cb)
            case "Range":
                log_conf.data_received_cb.add_callback(rec.log_range_cb)
            case "Battery":
                log_conf.data_received_cb.add_callback(rec.log_battery_cb) #(rec.battery_temp_cb)
            case "Sys":
                log_conf.data_received_cb.add_callback(rec.log_sys_cb)
            case "Supervisor":
                log_conf.data_received_cb.add_callback(rec.log_supervisor_cb)

        log_conf.error_cb.add_callback(rec.log_error_cb)
    else:
        print("One or more of the variables in the configuration was not found in log TOC. No logging will be possible.")

|- Loading State
|- Loading Range
|- Loading Battery


In [20]:
#next(item["name"] for item in log_configs if item.name == "franco" or item["name"] == "ciccio")

## 3.4 Pre-Flight Checks

In [21]:
# ---[ Pre configuration ]---
# Pre checks on Crazyflie deck, sensors..

"""
def information_callback(text: str):
    print(f"|- [Platform]: {text}", end='')
scf.cf.platform.fetch_platform_informations(information_callback)
print("CRTP Protocol Version: ", scf.cf.platform.get_protocol_version())
"""
# Console configuration
def console_callback(text: str):
    if text.startswith('SUP'):
        print(f"|- [Console]: {text}", end='')
    else:
        print(f"{text}", end='')
scf.cf.console.receivedChar.add_callback(console_callback)

# Flow deck checks callback
def param_deck_flow(_, value_str):
    global deck_attached_event
    # The flow deck that you are using, should be correctly attached to the crazyflie. 
    # If it is not, it will try to fly anyway without a good position estimate and for sure is going to crash. 

    #print(deck_attached_event)
    value = int(value_str)
    if value:
        deck_attached_event.set()
        print(f"|- [Deck check]: {_} is attached!: {value} ", end='')
    else:
        print(f"|- [Deck check]: {_} is NOT attached!: {value} ", end='')
    
scf.cf.param.add_update_callback(group="deck", name="bcFlow2", cb=param_deck_flow)
scf.cf.param.request_param_update('deck.bcFlow2')

if not deck_attached_event.wait(timeout=5):
    #print(deck_attached_event)
    print(f"|- [Deck check]: No flow deck detected!")
    # sys.exit(1)
else:
    print(deck_attached_event)
    print(f"|- [Deck check]: Ok")

# Sensor checks
# IMU - Inertial Measurement Unit
def param_imu_sensors(_, value_str):
    group, name = _.split(".")
    sensors = {"BMP3XX":"Barometer", "AK8963":"Magnetometer", "LPS25H":"Barometer"}
    value = int(value_str)
    if value:
        print(f"|- [IMU Sensors]: {sensors[name]} {name} is present: {value}")
    else:
        print(f"|- [IMU Sensors]: {sensors[name]} {name} is NOT present: {value}")

scf.cf.param.add_update_callback(group="imu_sensors", name="BMP3XX", cb=param_imu_sensors)
scf.cf.param.add_update_callback(group="imu_sensors", name="AK8963", cb=param_imu_sensors)
scf.cf.param.add_update_callback(group="imu_sensors", name="LPS25H", cb=param_imu_sensors)
scf.cf.param.request_param_update('imu_sensors.BMP3XX')
scf.cf.param.request_param_update('imu_sensors.AK8963')
scf.cf.param.request_param_update('imu_sensors.LPS25H')

# Stabilizer - Estimator checks
def param_estimator(_, value_str):
    group, name = _.split(".")
    estimator_type = { 0:"Auto select", 1:"Complementary", 2:"Extended Kalman", 3:"Unscented Kalman" }
    value = int(value_str)
    print(f"|- [Parameter] {group} - {name}: \"{estimator_type[value]} ({value})\" is used!")
scf.cf.param.add_update_callback(group="stabilizer", name="estimator", cb=param_estimator)

# Stabilizer - Controller checks - https://www.bitcraze.io/documentation/repository/crazyflie-firmware/master/functional-areas/sensor-to-control/controllers/
def param_controller(_, value_str):
    group, name = _.split(".")
    controller_type = {0:"Auto select", 1:"PID", 2:"Mellinger", 3:"INDI", 4:"Brescianini", 5:"Lee"}
    value = int(value_str)
    print(f"|- [Parameter] {group} - {name}: \"({value}) - {controller_type[value]}\" is used!")
    
scf.cf.param.add_update_callback(group="stabilizer", name="controller", cb=param_controller)

# State checks
scf.cf.param.set_value('supervisor.infdmp', '1') # When nonzero, dump information about the current supervisor state to the console log

time.sleep(1)

|- [Deck check]: deck.bcFlow2 is attached!: 1 <threading.Event at 0x7f52259280: set>
|- [Deck check]: Ok
|- [IMU Sensors]: Barometer BMP3XX is present: 1
|- [IMU Sensors]: Magnetometer AK8963 is NOT present: 0
|- [IMU Sensors]: Barometer LPS25H is NOT present: 0
|- [Console]: SUP: Supervisor info ---
|- [Console]: SUP: State: Ready to fly
|- [Console]: SUP: Conditions: (0x401)
|- [Console]: SUP:   armed (0x1): 1
|- [Console]: SUP:   isFlying (0x2): 0
|- [Console]: SUP:   isTumbled (0x4): 0
|- [Console]: SUP:   commanderWdtWarning (0x8): 0
|- [Console]: SUP:   commanderWdtTimeout (0x10): 0
|- [Console]: SUP:   emergencyStop (0x20): 0
|- [Console]: SUP:   isCrashed (0x40): 0
|- [Console]: SUP:   preflightTimeout (0x80): 0
|- [Console]: SUP:   landingTimeout (0x100): 0
|- [Console]: SUP:   deckFault (0x200): 0
|- [Console]: SUP:   rpmAtArmingValid (0x400): 1
|- [Console]: SUP:   spinupTimeout (0x800): 0
|- [Console]: SUP:   motorsNotResponding (0x1000): 0


## 3.5 Estimator and Controller

In [22]:
# Tip: It is better to set the Stabilizer Estimator before setting the Stabilizer Controller
# Stabilizer Estimator selection : 0:"Auto select", 1:"Complementary", 2:"Extended Kalman", 3:"Unscented Kalman"

# If Flow Deck is used
if deck_attached_event.is_set():
    scf.cf.param.set_value('stabilizer.estimator', str(2)) # Choose Extended Kalman filter
else:
    scf.cf.param.set_value('stabilizer.estimator', str(1)) # Else choose Complementary filter, recommended for manual control

# Stabilizer Controller selection : 1:"PID", 2:"Mellinger", 3:"INDI", 4:"Brescianini", 5:"Lee"
scf.cf.param.set_value('stabilizer.controller', str(0))

scf.cf.param.get_value("hlCommander.vtoff", timeout=60)

# Reset estimator
reset_estimator(scf) # resets the Kalman filter and makes the Crazyflie wait until it has an accurate position estimate
time.sleep(1)

|- [Parameter] stabilizer - estimator: "Extended Kalman (2)" is used!
|- [Parameter] stabilizer - controller: "(0) - Auto select" is used!
CONTROLLER: Using PID (1) controller
Waiting for estimator to find position...success!


## 3.6 Commander controller setup

In [23]:
# ---[ Commander ]---
#commander(scf)
print("-"*30 + "\n COMMANDER CONTROL \n" + "-"*30)
global dev
global endpoint

relative_yaw=False

# Commander
global commander
commander = scf.cf.commander
commander.set_client_xmode(enabled=True)

# Supervisor
global sup
sup = scf.cf.supervisor

# Arm the Crazyflie
#scf.cf.supervisor.send_arming_request(True)
#scf.cf.platform.send_arming_request(do_arm=True)
#time.sleep(1.0)

takeoff_yaw = 3.14 / 2 if relative_yaw else 0.0

# External Position Sender
position = scf.cf.extpos
recovery = False
stop = False
print("READ BITFIELD", sup.read_bitfield())
print("READ BITFIELD", sup.read_state_list())

# Arm the Crazyflie
while not ('Can be armed' in sup.read_state_list()): # Can be armed ?
    if 'Is armed' in sup.read_state_list():
        break
    time.sleep(0.3)
    if bool('Is locked' in scf.cf.supervisor.read_state_list()):
        print("|- [Supervisor] - Crazyflie State: Is locked")
    continue
    
# DEPRECATED - scf.cf.platform.send_arming_request(True)
sup.send_arming_request(do_arm=True) # ARM
print("|- [Commander] - Drone ARMED!")

# Check Arm
while not ('Is armed' in sup.read_state_list()): # Is armed ?
    if bool('Is armed' in scf.cf.supervisor.read_state_list()):
        print("|- [Supervisor] - Crazyflie State: Is armed")
    time.sleep(0.3)
    print("b")
    continue

crazyflie_ready.set()
print("|- [Commander] - Drone IS ARMED!")

# Wait until Can fly
while not ('Can fly' in sup.read_state_list()): # Can fly ?
    time.sleep(0.3)
    print("c")
    continue

print("|- [Supervisor] - Can fly: ", bool('Can fly' in sup.read_state_list()))
print("|- [Commander] - Drone CAN FLY!")

# min = 10001 , mid = 30001, max = 60000 
#print(f"Is Flying: {bool(sup.decode_bitfield(4))}", end="")

------------------------------
 COMMANDER CONTROL 
------------------------------
READ BITFIELD 526
READ BITFIELD ['Is armed', 'Is auto armed', 'Can fly', 'Finished HL trajectory']
|- [Commander] - Drone ARMED!
|- [Commander] - Drone IS ARMED!
|- [Supervisor] - Can fly:  True
|- [Commander] - Drone CAN FLY!


## 3.7 Start logging

In [24]:
for log_conf in log_configs:
    log_conf.start()

In [ ]:
#from datetime import datetime
#date_object = datetime.fromtimestamp(250448)
#date_object

## 3.8 Mission

In [26]:
s = {"stop_reason": "distance"}
s["vbat_start_v"] = rec.battery
s["batt_start_pct_est"] = rec.vbat_to_pct(s["vbat_start_v"])

In [27]:
"""Take off, fly `distance` metres forward, land. Log to csv_path.
    Returns a dict with the run summary."""

'Take off, fly `distance` metres forward, land. Log to csv_path.\n    Returns a dict with the run summary.'

In [28]:
# ---[ Mission ]---
def mission(scf, configs, recorder):
    distance = 0.7 # meters
    velocity = 0.2 # meters/second
    height = 0.3 # meters
    min_front_mm = 300 # millimeters
    global s
    
    try:
        # MotionCommander lands automatically when leaving the 'with'
        with MotionCommander(scf, default_height=height) as mc:
            rec.phase = "hover"
            time.sleep(2.0)

            x0, y0, yaw0 = rec.x, rec.y, rec.yaw
            rec.phase = "forward"
            t_start = time.monotonic()
            timeout = 2 * distance / velocity + 2
            mc.start_forward(velocity=velocity)
            while True:
                if rec.x - x0 >= distance:
                    break
                front = rec.ranges.get("range.front")
                if front is not None and front < min_front_mm:
                    s["stop_reason"] = "obstacle"
                    break
                if time.monotonic() - t_start > timeout:
                    s["stop_reason"] = "timeout"
                    break
                time.sleep(0.02)
            s["forward_time_s"] = round(time.monotonic() - t_start, 2)
            mc.stop()
            s["dist_at_stop_cm"] = round((rec.x - x0) * 100, 1)

            rec.phase = "settle"
            time.sleep(1.5)
            rec.phase = "landing"
            s["dist_final_cm"] = round((rec.x - x0) * 100, 1)
            s["drift_cm"] = round((rec.y - y0) * 100, 1)
            s["yaw_change_deg"] = round(rec.yaw - yaw0, 1)
            rec.phase = "landing"
        rec.phase = "landed"
        time.sleep(1.0)
        vb = rec.ranges.get("pm.vbat")
        s["vbat_end_v"] = rec.battery
        s["batt_end_pct_est"] = rec.vbat_to_pct(s["vbat_end_v"])
    finally: 
        for log_conf in log_configs:
            log_conf.stop()
        scf.close_link()
        recorder.close()
        print(f"Saved {path}")
        time.sleep(0.2)

mission(scf, log_configs, rec)
#trajectory_id = 1
#duration = mission(scf, trajectory_id)

Saved logs/flight_20261016_172946.csv


# 4. Commander

## 4.1 High Level Commander

In [ ]:
commander_hl = scf.cf.high_level_commander

In [ ]:
reset_estimator(scf)

In [ ]:
commander_hl.takeoff(0.3, duration_s=0.1)

In [ ]:
commander_hl.land(absolute_height_m=0.0, duration_s=1.2)

In [ ]:
commander_hl.spiral(
    angle=-math.pi * 2.0,
    r0=0.50, rF=0.5, 
    ascent=0.2, duration_s=4.0, 
    sideways=True, clockwise=True
)

## 4.2 - Position High Level Commander

In [ ]:
# High Level Commander
hl_commander = PositionHlCommander(
    crazyflie=scf, 
    x=0.0, y=0.0, z=0.0, 
    default_velocity=0.2, 
    default_height=0.3, controller=PositionHlCommander.CONTROLLER_PID, 
    default_landing_height=0.0)

print(f"|- [HL Commander] - Current position: {hl_commander.get_position()}")

#hl_commander.forward(0.2)
#hl_commander.left(0.2)
#hl_commander.back(0.2)
#hl_commander.go_to(0.0, 0.0, 0.4)

In [ ]:
hl_commander.take_off()

In [ ]:
hl_commander.forward()

## 4.3 - Motion Commander

In [ ]:
mc = False
if not mc:
    print("|- 4. MOTION COMMANDER CONTROL", end='\n|-----------------------------')
    mc = MotionCommander(scf.cf, default_height=0.0)
else:
    pass

In [ ]:
forward_bt = widgets.Button(description='Forward', disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='F',
    icon='arrow-up' # (FontAwesome names without the `fa-` prefix)
)
backward_bt = widgets.Button(description='Backward', disabled=False, button_style='', tooltip='B', icon='arrow-down')
left_bt = widgets.Button(description='Left', disabled=False, button_style='', tooltip='L', icon='arrow-left')
right_bt = widgets.Button(description='Right', disabled=False, button_style='', tooltip='R', icon='arrow-right')
land_bt = widgets.Button(description='Land', disabled=False, button_style='', tooltip='S')

height_m = widgets.FloatSlider(value=0.4, min=-0.2, max=0.7, step=0.02, description='Height:', disabled=False, continuous_update=True,
                               orientation='vertical', readout=True, readout_format='f')

def move_update(b):
    global move
    global stop

    match(b.tooltip):
        case "F":
            move[0] = True
        case "B":
            move[1] = True
        case "L":
            move[2] = True
        case "R":
            move[3] = True
        case "S":
            stop = True
    with output:
        print(move)
            
forward_bt.on_click(move_update)
backward_bt.on_click(move_update)
left_bt.on_click(move_update)
right_bt.on_click(move_update)

In [ ]:
mc.take_off(height=0.4, velocity=0.5)

In [ ]:
def mc_sender(name, commander, sup):
    distance = 0.05
    print(f"|- [Motion Commander Sender] Name: {name}, Distance: {distance}")
    global move
    global distance_int
    global position_estimate
    global stop

    # Send thrust=0 first so that the crazyflie-firmware's safety protection requirements are met (https://forum.bitcraze.io/viewtopic.php?t=4574)
    #commander.send_setpoint(0.0, 0.0, 0, 0)
    #time.sleep(0.01)
    commander.take_off(height=height_m.value, velocity=0.2)
    time.sleep(2.0)
    
    states = sup.read_state_list()
    while 'Can fly' in states:
        states = sup.read_state_list()
        try:
            print(f"|- [Motion Commander] - Is flying : Height: {height_m.value}, Position estimate: [{position_estimate[0]:.5f},{position_estimate[1]:.5f}]")
            if not any(move):
                sup.send_emergency_stop_watchdog()
                #commander.move_distance(0.0, 0.0, 0.0, velocity=0.2)
                """
                if distance_int.value > 0.0:
                    commander.up(distance_int.value, velocity=0.2)
                if distance_int.value < 0.0:
                    commander.down(-distance_int.value, velocity=0.2)
                """
            else:
                if move[0]: # Forward
                    print("Move Front")
                    commander.move_distance(distance, 0.0, 0.0, velocity=0.3)
                    #commander.front(distance, velocity=0.2)
                    move[0] = False
                elif move[1]: # Backward
                    print("Move Back")
                    commander.move_distance(-distance, 0.0, 0.0, velocity=0.3)
                    #commander.back(distance, velocity=0.2)
                    move[1] = False
                elif move[2]: # Left
                    print("Move Left")
                    commander.move_distance(0.0, distance, 0.0, velocity=0.3)
                    #commander.left(distance, velocity=0.2)
                    move[2] = False
                elif move[3]: # Right
                    print("Move Right")
                    commander.move_distance(0.0, -distance, 0.0, velocity=0.3)
                    #commander.right(distance, velocity=0.2)
                    move[3] = False
            if stop:
                raise KeyboardInterrupt
                
            time.sleep(0.01)
            
        except KeyboardInterrupt: # Let CTRL+C actually exit
            sup.send_emergency_stop_watchdog()
            sup.send_emergency_stop() # Emergency Stop Command
            print("|- [Supervisor] - Emergency Stop Command sent!")
            break
            
        except Exception as e:
            print(e)
            commander.land()
            #print(f"*", end=' ')
            #commander.send_setpoint(roll, pitch, yawrate, thrust_int)
            #commander.send_setpoint_manual(0, 0, 0, 20.0, rate=False)
            time.sleep(0.01)
            #commander.send_setpoint(roll, pitch, yawrate, thrust)
            break
            
        sup.send_emergency_stop_watchdog()
    
x = threading.Thread(target=mc_sender, args=(1, mc, sup), daemon=True)
#logging.info("Main    : before running thread")
x.start()

In [ ]:
output = widgets.Output()
display(widgets.HBox([height_m, forward_bt, backward_bt, left_bt, right_bt]), output)

In [ ]:
stop = True
x.join()

In [ ]:
print(f"|- [Motion Commander] - Is flying : Height: {height_m.value}, Position estimate: [{position_estimate[0]:.5f},{position_estimate[1]:.5f}]")
#widgets.HBox([forward_bt, backward_bt, left_bt, right_bt])

In [ ]:
mc.land()
for log_conf in log_configs:
    log_conf.stop()

In [ ]:
scf.close_link()

In [ ]:
mc.start_linear_motion(
    velocity_x_m=velocity_x, 
    velocity_y_m=velocity_y, 
    velocity_z_m=0,
    rate_yaw=0
)

In [ ]:
stop = True

## 4.4 - Base Commander

In [ ]:
roll    = widgets.FloatSlider(value=0.0, min=-20.0, max=20.0, step=1.0, description='Roll:', disabled=False, continuous_update=True, 
                              orientation='horizontal', readout=True, readout_format='.1f',)
pitch   = widgets.FloatSlider(value=0.0, min=-20.0, max=20.0, step=1.0, description='Pitch:', disabled=False, continuous_update=True,
                              orientation='vertical', readout=True, readout_format='.1f',)
yawrate = widgets.IntSlider(value=0, min=-180, max=180, step=10, description='Yawrate:', disabled=False, continuous_update=True,
                            orientation='horizontal', readout=True, readout_format='d')
thrust  = 20.0
thrust_int = widgets.IntSlider(value=10001, min=10001, max=50001, step=100, description='Thrust:', disabled=False, continuous_update=True,
                               orientation='vertical', readout=True, readout_format='d')
stop = False
def emergency_stop():
    global stop
    stop = True
    return
    
button = widgets.Button(
    description='Emergency Stop',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Emergency Stop',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)
button.on_click(emergency_stop)

In [ ]:
"""
# Send thrust=0 first so that the crazyflie-firmware's safety protection requirements are met (https://forum.bitcraze.io/viewtopic.php?t=4574)
commander.send_setpoint(0.0, 0.0, 0, 0)
time.sleep(0.01)

# Send thrust of 30000 for 3 seconds
for i in range(300):
    commander.send_setpoint(roll, pitch, yawrate, thrust_int)
    time.sleep(0.01)  # send the setpoint once every 10 ms as per the user guide
commander.send_stop_setpoint()
"""

In [ ]:
#commander.send_stop_setpoint()
def watchdog_sender(name, commander, sup):
    print(f"|- [Watchdog Sender] Name: {name}, Thrust: {thrust}")
    global roll
    global pitch
    global yawrate
    global thrust_int
    global stop
    
    # Send thrust=0 first so that the crazyflie-firmware's safety protection requirements are met (https://forum.bitcraze.io/viewtopic.php?t=4574)
    commander.send_setpoint(0.0, 0.0, 0, 0)
    time.sleep(0.01)
    
    # Send thrust of 30000 for 3 seconds
    #logging.info("Thread %s: start feed setpoint", name)
    print(f"|- [Watchdog Sender] ", end=' ')
    while 'Can fly' in sup.read_state_list():
        try:
            #print(f".", end=' ')
            commander.send_setpoint(roll.value, pitch.value, yawrate.value, thrust_int.value)
            #commander.send_setpoint_manual(0, 0, 0, 20.0, rate=False)
            time.sleep(0.01)
            if stop:
                raise KeyboardInterrupt
            
        except KeyboardInterrupt: # Let CTRL+C actually exit
            sup.send_emergency_stop_watchdog()
            sup.send_emergency_stop() # Emergency Stop Command
            print("|- [Supervisor] - Emergency Stop Command sent!")
            break
            
        except:
            print(f"*", end=' ')
            commander.send_setpoint(roll, pitch, yawrate, thrust_int)
            #commander.send_setpoint_manual(0, 0, 0, 20.0, rate=False)
            time.sleep(0.01)
            #commander.send_setpoint(roll, pitch, yawrate, thrust)
            pass
            
        sup.send_emergency_stop_watchdog()
    
    #logging.info("Thread %s: finishing", name)

x = threading.Thread(target=watchdog_sender, args=(1, commander, sup), daemon=True)
#logging.info("Main    : before running thread")
x.start()

In [ ]:
widgets.HBox([thrust_int, pitch, roll, yawrate, button])

In [ ]:
stop = True

In [ ]:
thrust_int = 40001

In [ ]:
sup.send_emergency_stop()

In [ ]:
commander.send_setpoint_manual(0, 0, 0, 100.0, rate=False)

# Fase di richiesta di un altro drone

In [ ]:
# 1. Parte la richiesta dall'edge 2 indirizzata verso edge 1
# 2. Richiesta elaborata e validata dal fog node
# 3. Richiesta inoltrata dal fog node all'edge 1 (o in generale a tutti gli edge che vogliono partecipare al contratto)
# 4. Edge 1 risponde con la disponibilità che ha
# 5. Fog node comunica la disponibilità dell'edge 1 con l'offerta
# 6. Edge 2 accetta l'offerta
# 7. Fog node riceve la risposta che l'offerta è accettata e dunque assegna un token al drone dell'Edge 1
# 8. Edge 1 riceve la comunicazione dal Fog node e assegna il token al drone, dunque lo invia verso la direzione dell'Edge 2
# 9. Edge 2 avvia la ricerca del nuovo drone alla flotta, appena è raggiungibile tramite l'indirizzo, ne valida il token.
# 10. Se il token è valido, il drone è aggiunto alla flotta ed è pronto ad eseguire nuovi task.

## Controllo drone con Joystick

In [ ]:
while 'Can fly' in sup.read_state_list(): # Is flying ?
    control = None
    try:
        control = dev.read(endpoint.bEndpointAddress, endpoint.wMaxPacketSize, USB_TIMEOUT)
        #commander.send_setpoint_manual(0, 0, 0, 20.0, True)
        #commander.send_velocity_world_setpoint(0.0, 0.0, 0.2, 0)
    
        match control[2]:
            case 82:
                pitch = 25
                commander.send_setpoint_manual(0, pitch, 0, thrust, rate=False)
            case 81:
                pitch = -25
                commander.send_setpoint_manual(0, pitch, 0, thrust, rate=False)
            case 80:
                roll = -25
                commander.send_setpoint_manual(roll, 0, 0, thrust, rate=False)
            case 79:
                roll = 25
                commander.send_setpoint_manual(roll, 0, 0, thrust, rate=False)
            case 4:
                print("THRUST INCREASE") # thrust + 1
                thrust += 2.5
            case 29:
                print("THRUST DECREASE") # thrust - 1
                thrust -= 2.5
            case 16:
                print("STOP WATCHDOG")
                sup.send_emergency_stop_watchdog()           
            case 17:
                print("CRASH RECOVERY REQUEST")
                sup.send_crash_recovery_request() # Use Ctrl + R to send a "Crash Recovery" request
            case 18:
                yawrate -= 20
                #commander.send_setpoint_manual(0, 0, yawrate, thrust, rate=False)
            case 19:
                yawrate += 20
                #commander.send_setpoint_manual(0, 0, yawrate, thrust, rate=False)
            case None:
                raise Exception
            case _: # wildcard - simile ad un else, deve stare alla fine
                print("Roll: ", roll, "Pitch: ", pitch, "Yaw: ", yawrate, "Thrust: ", thrust)
                roll = 0
                pitch = 0
    
        # Position
        #commander.send_setpoint(0, 0, 0, thrust) # roll, pitch, yawrate, thrust
        #commander.send_position_setpoint(0.0, 0.0, 0.2, 0)
        #commander.send_velocity_world_setpoint(0.0, 0.0, 0.2, 0)
    
        # Distance
        # - Only with Flow Deck for Z distance from the ground.
        #commander.send_hover_setpoint(0.0, 0.0, 0.0, 0.1) # vx, vy, yawrate, zdistance
    
        # Land
        #commander.send_notify_setpoint_stop(remain_valid_milliseconds=0)
        #print(".", end="")
    
    except KeyboardInterrupt: # Let CTRL+C actually exit
        sup.send_emergency_stop_watchdog()
        sup.send_emergency_stop() # Emergency Stop Command
        print("|- [Commander] - Emergency Stop Command sent!")
        break
    
    except:
        #commander.send_setpoint(0, 0, 0, int(0.5 * 1000))
        commander.send_setpoint_manual(0, 0, 0, thrust, rate=False)
        #commander.send_setpoint_manual(0, 0, yawrate, thrust, rate=False) # roll, pitch, yawrate, thrust_percentage, rate
        pass
    
    sup.send_emergency_stop_watchdog()
    time.sleep(0.05)
    
    print("Is flying: ", bool('Is flying' in sup.read_state_list()))
    if bool('Is flying' in sup.read_state_list()):
    print("|- [Commander] - Drone TAKEOFF!")
    print("|- [Commander] - Drone IS FLYING!")
print("|- -------- Main - END ------- ")